In [1]:
import sys
import os

# Add the src directory to sys.path
sys.path.append(os.path.abspath("."))

In [2]:
import scipy.io
import os
import numpy as np
import trimesh

# Path to the Chair folder
chair_dir = './dropbox/Chair/'

# Example model index
model_idx = 1

In [3]:
import scipy.io
import os

chair_dir = './dropbox/Chair/'
model_idx = 1

# List of subfolders to check
subfolders = ['ops', 'labels', 'boxes', 'syms', 'part mesh indices']

for folder in subfolders:
    file_path = os.path.join(chair_dir, folder, f'{model_idx}.mat')
    if os.path.exists(file_path):
        mat_data = scipy.io.loadmat(file_path)
        print(f"\n--- Keys in {folder}/{model_idx}.mat ---")
        for key in mat_data.keys():
            print(key, type(mat_data[key]), end=' ')
            try:
                print(mat_data[key].shape)
            except:
                print()
    else:
        print(f"{folder}/{model_idx}.mat not found")


--- Keys in ops/1.mat ---
__header__ <class 'bytes'> 
__version__ <class 'str'> 
__globals__ <class 'list'> 
op <class 'numpy.ndarray'> (1, 5)

--- Keys in labels/1.mat ---
__header__ <class 'bytes'> 
__version__ <class 'str'> 
__globals__ <class 'list'> 
label <class 'numpy.ndarray'> (1, 3)

--- Keys in boxes/1.mat ---
__header__ <class 'bytes'> 
__version__ <class 'str'> 
__globals__ <class 'list'> 
box <class 'numpy.ndarray'> (12, 3)

--- Keys in syms/1.mat ---
__header__ <class 'bytes'> 
__version__ <class 'str'> 
__globals__ <class 'list'> 
shapename <class 'numpy.ndarray'> (1,)
sym <class 'numpy.ndarray'> (8, 1)

--- Keys in part mesh indices/1.mat ---
__header__ <class 'bytes'> 
__version__ <class 'str'> 
__globals__ <class 'list'> 
cell_boxs_correspond_objSerialNumber <class 'numpy.ndarray'> (1, 3)
shapename <class 'numpy.ndarray'> (1,)


In [4]:
import scipy.io
import os

chair_dir = './dropbox/Chair/'
model_idx = 1

# Load node types
ops_data = scipy.io.loadmat(os.path.join(chair_dir, 'ops', f'{model_idx}.mat'))['op']
print("Ops data:", ops_data)

# Load labels
labels_data = scipy.io.loadmat(os.path.join(chair_dir, 'labels', f'{model_idx}.mat'))['label']
print("Labels data:", labels_data)

# Load boxes
boxes_data = scipy.io.loadmat(os.path.join(chair_dir, 'boxes', f'{model_idx}.mat'))['box']
print("Boxes data:", boxes_data)

# Load symmetries
syms_data = scipy.io.loadmat(os.path.join(chair_dir, 'syms', f'{model_idx}.mat'))['sym']
print("Syms data:", syms_data)

# Load part mesh indices
part_mesh_idx_data = scipy.io.loadmat(os.path.join(chair_dir, 'part mesh indices', f'{model_idx}.mat'))['cell_boxs_correspond_objSerialNumber']
print("Part mesh indices:", part_mesh_idx_data)

Ops data: [[0 0 1 0 1]]
Labels data: [[1 2 0]]
Boxes data: [[ 0.0030865   0.00419755  0.0302842 ]
 [ 0.148677   -0.422335    0.488309  ]
 [ 0.0724135   0.169623   -0.315397  ]
 [ 0.223845    0.523685    0.781198  ]
 [ 0.805681    1.15351     0.305279  ]
 [ 0.890567    0.898086    0.89214   ]
 [ 0.         -0.0121707   0.0994548 ]
 [ 1.          0.770206    0.981659  ]
 [ 0.         -0.637679   -0.162646  ]
 [ 0.         -0.00438424 -0.00601635]
 [ 0.          0.637679    0.164046  ]
 [ 1.          0.77029     0.986434  ]]
Syms data: [[0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]]
Part mesh indices: [[array([[5, 6, 7, 8]], dtype=uint8) array([[ 9, 10]], dtype=uint8)
  array([[1, 2, 3, 4]], dtype=uint8)]]


In [5]:
leaf_indices = [i for i, t in enumerate(ops_data[0]) if t == 0]

print("Leaf node info:")
for idx, leaf_idx in enumerate(leaf_indices):
    node_type = ops_data[0][leaf_idx]
    label = labels_data[0][idx]
    box = boxes_data[leaf_idx]
    mesh_indices = part_mesh_idx_data[0][idx][0]  # unwrap array
    print(f"Leaf node {leaf_idx}: type={node_type}, label={label}, box={box}, mesh indices={mesh_indices}")

Leaf node info:
Leaf node 0: type=0, label=1, box=[0.0030865  0.00419755 0.0302842 ], mesh indices=[5 6 7 8]
Leaf node 1: type=0, label=2, box=[ 0.148677 -0.422335  0.488309], mesh indices=[ 9 10]
Leaf node 3: type=0, label=0, box=[0.223845 0.523685 0.781198], mesh indices=[1 2 3 4]


In [6]:
import anytree
from anytree import Node, RenderTree

# Build nodes
nodes = []
for i, t in enumerate(ops_data[0]):
    label = None
    if t == 0:  # leaf node
        leaf_idx = leaf_indices.index(i)
        label = labels_data[0][leaf_idx]
    nodes.append(Node(f"Node{i} type={t} label={label}"))

# Set parent-child relationships (depth-first assumption)
# ops = [0, 0, 1, 0, 1] -> simple example
nodes[2].children = (nodes[0], nodes[1])  # adjacency node 2 has children 0 and 1
nodes[4].children = (nodes[2], nodes[3])  # adjacency node 4 has children 2 and 3

# Print tree
for pre, fill, node in RenderTree(nodes[4]):
    print(f"{pre}{node.name}")

Node4 type=1 label=None
├── Node2 type=1 label=None
│   ├── Node0 type=0 label=1
│   └── Node1 type=0 label=2
└── Node3 type=0 label=0


In [7]:
import k3d
import numpy as np

# Create a K3D plot
plot = k3d.plot(grid_visible=True)

# Leaf node indices
leaf_indices = [0, 1, 3]

# Boxes centers from your data
boxes = boxes_data[leaf_indices]

# Size for visualization (you can adjust this if needed)
box_size = 0.1

# Function to create a cube mesh for K3D
def make_cube(center, size, color=0xff0000):
    x, y, z = center
    # 8 corners of the cube
    corners = np.array([
        [x-size/2, y-size/2, z-size/2],
        [x+size/2, y-size/2, z-size/2],
        [x+size/2, y+size/2, z-size/2],
        [x-size/2, y+size/2, z-size/2],
        [x-size/2, y-size/2, z+size/2],
        [x+size/2, y-size/2, z+size/2],
        [x+size/2, y+size/2, z+size/2],
        [x-size/2, y+size/2, z+size/2],
    ], dtype=np.float32)

    # 12 triangles (2 per face)
    indices = np.array([
        0,1,2, 0,2,3,  # bottom
        4,5,6, 4,6,7,  # top
        0,1,5, 0,5,4,  # front
        2,3,7, 2,7,6,  # back
        1,2,6, 1,6,5,  # right
        0,3,7, 0,7,4   # left
    ], dtype=np.uint32)

    return k3d.mesh(corners, indices, color=color, opacity=0.5)

# Add each leaf node box to the plot
colors = [0xff0000, 0x00ff00, 0x0000ff]  # different colors per box
for i, center in enumerate(boxes):
    plot += make_cube(center, box_size, colors[i])

plot.display()

Output()

In [8]:
import k3d
import numpy as np
import os

def read_obb_12(file_path):
    """
    Reads an OBB file where each line has 12 numbers:
    center_x, center_y, center_z, length_x, length_y, length_z,
    dir1_x, dir1_y, dir1_z, dir2_x, dir2_y, dir2_z
    """
    obbs = []
    with open(file_path, 'r') as f:
        for line in f:
            nums = list(map(float, line.strip().split()))
            if len(nums) == 12:
                center = np.array(nums[0:3])
                lengths = np.array(nums[3:6])
                dir1 = np.array(nums[6:9])
                dir2 = np.array(nums[9:12])
                dir1 /= np.linalg.norm(dir1)
                dir2 /= np.linalg.norm(dir2)
                dir3 = np.cross(dir1, dir2)
                dir3 /= np.linalg.norm(dir3)
                obbs.append({'center': center, 'lengths': lengths, 'dir1': dir1, 'dir2': dir2, 'dir3': dir3})
    return obbs

def make_oriented_box_k3d(obb, color=0xff0000):
    c = obb['center']
    l = obb['lengths']
    d1, d2, d3 = obb['dir1'], obb['dir2'], obb['dir3']

    d1 *= l[0]/2
    d2 *= l[1]/2
    d3 *= l[2]/2

    corners = np.array([
        c - d1 - d2 - d3,
        c - d1 + d2 - d3,
        c + d1 - d2 - d3,
        c + d1 + d2 - d3,
        c - d1 - d2 + d3,
        c - d1 + d2 + d3,
        c + d1 - d2 + d3,
        c + d1 + d2 + d3,
    ], dtype=np.float32)

    indices = np.array([
        0,1,2, 0,2,3,  # bottom
        4,5,6, 4,6,7,  # top
        0,1,5, 0,5,4,  # front
        2,3,7, 2,7,6,  # back
        1,2,6, 1,6,5,  # right
        0,3,7, 0,7,4   # left
    ], dtype=np.uint32)

    return k3d.mesh(corners, indices, color=color, opacity=0.3)

# Load OBBs
model_idx = 1
obb_file = os.path.join('./dropbox/Chair/obbs', f'{model_idx}.obb')
obbs = read_obb_12(obb_file)

# Create K3D plot
plot = k3d.plot(grid_visible=True)
colors = [0xff0000, 0x00ff00, 0x0000ff, 0xffff00, 0xff00ff, 0x00ffff]

for i, obb in enumerate(obbs):
    plot += make_oriented_box_k3d(obb, color=colors[i % len(colors)])

plot.display()

FileNotFoundError: [Errno 2] No such file or directory: './dropbox/Chair/obbs/1.obb'

In [ ]:
import scipy.io
ops_data = scipy.io.loadmat('./dropbox/Chair/ops/1.mat')
ops_data

{'__header__': b'MATLAB 5.0 MAT-file, Platform: PCWIN64, Created on: Tue Oct 15 15:09:20 2019',
 '__version__': '1.0',
 '__globals__': [],
 'op': array([[0, 0, 1, 0, 1]], dtype=uint8)}

In [ ]:
import scipy.io
syms_mat = scipy.io.loadmat('./dropbox/Chair/syms/1.mat')
partnet_id = int(syms_mat['shapename'][0])
print("PartNet ID:", partnet_id)  # should be 172

PartNet ID: 1095


In [ ]:
import os

partnet_id = 1095  # from syms_mat['shapename'][0]

# Paths
model_file = f'./dropbox/Chair/models/{partnet_id}.obj'
obb_file   = f'./dropbox/Chair/obbs/{partnet_id}.obb'
ops_file  = './dropbox/Chair/ops/1.mat'
labels_file = './dropbox/Chair/labels/1.mat'
boxes_file = './dropbox/Chair/boxes/1.mat'
part_mesh_file = './dropbox/Chair/part mesh indices/1.mat'
syms_file = './dropbox/Chair/syms/1.mat'

print("Model path:", model_file)
print("OBB path:", obb_file)

Model path: ./dropbox/Chair/models/1095.obj
OBB path: ./dropbox/Chair/obbs/1095.obb


In [ ]:
import k3d
import numpy as np
import scipy.io
import os

# ----------------------
# 1. Set internal index
# ----------------------
internal_idx = 1  # e.g., 1.mat
chair_dir = './dropbox/Chair/'

# ----------------------
# 2. Load PartNet ID
# ----------------------
syms_mat = scipy.io.loadmat(os.path.join(chair_dir, 'syms', f'{internal_idx}.mat'))
partnet_id = int(syms_mat['shapename'][0])
print("PartNet ID:", partnet_id)

# ----------------------
# 3. Load OBB file
# ----------------------
obb_file = os.path.join(chair_dir, 'obbs', f'{partnet_id}.obb')

def read_obb_12(file_path):
    """
    Reads a 12-element OBB file:
    center_x, center_y, center_z, length_x, length_y, length_z,
    dir1_x, dir1_y, dir1_z, dir2_x, dir2_y, dir2_z
    """
    obbs = []
    with open(file_path, 'r') as f:
        for line in f:
            nums = list(map(float, line.strip().split()))
            if len(nums) == 12:
                center = np.array(nums[0:3])
                lengths = np.array(nums[3:6])
                dir1 = np.array(nums[6:9]); dir1 /= np.linalg.norm(dir1)
                dir2 = np.array(nums[9:12]); dir2 /= np.linalg.norm(dir2)
                dir3 = np.cross(dir1, dir2); dir3 /= np.linalg.norm(dir3)
                obbs.append({'center': center, 'lengths': lengths, 'dir1': dir1, 'dir2': dir2, 'dir3': dir3})
    return obbs

def make_oriented_box_k3d(obb, color=0xff0000):
    c = obb['center']
    l = obb['lengths']
    d1, d2, d3 = obb['dir1'], obb['dir2'], obb['dir3']
    d1 *= l[0]/2; d2 *= l[1]/2; d3 *= l[2]/2
    corners = np.array([
        c - d1 - d2 - d3,
        c - d1 + d2 - d3,
        c + d1 - d2 - d3,
        c + d1 + d2 - d3,
        c - d1 - d2 + d3,
        c - d1 + d2 + d3,
        c + d1 - d2 + d3,
        c + d1 + d2 + d3,
    ], dtype=np.float32)
    indices = np.array([
        0,1,2, 0,2,3,
        4,5,6, 4,6,7,
        0,1,5, 0,5,4,
        2,3,7, 2,7,6,
        1,2,6, 1,6,5,
        0,3,7, 0,7,4
    ], dtype=np.uint32)
    return k3d.mesh(corners, indices, color=color, opacity=0.3)

# Read OBBs
obbs = read_obb_12(obb_file)

# ----------------------
# 4. Plot in K3D
# ----------------------
plot = k3d.plot(grid_visible=True)
colors = [0xff0000, 0x00ff00, 0x0000ff, 0xffff00, 0xff00ff, 0x00ffff]

for i, obb in enumerate(obbs):
    plot += make_oriented_box_k3d(obb, color=colors[i % len(colors)])

plot.display()

PartNet ID: 1095


ValueError: could not convert string to float: 'N'

In [ ]:
import k3d
import numpy as np
import os

def read_obb_file(file_path):
    """Read OBBs in 12-element format, same as draw3dobb.py"""
    obbs = []
    with open(file_path, 'r') as f:
        for line in f:
            vals = list(map(float, line.strip().split()))
            if len(vals) == 12:
                center = np.array(vals[0:3])
                lengths = np.array(vals[3:6])
                dir1 = np.array(vals[6:9]); dir1 /= np.linalg.norm(dir1)
                dir2 = np.array(vals[9:12]); dir2 /= np.linalg.norm(dir2)
                dir3 = np.cross(dir1, dir2); dir3 /= np.linalg.norm(dir3)
                obbs.append({'center': center, 'lengths': lengths, 'dir1': dir1, 'dir2': dir2, 'dir3': dir3})
    return obbs

def make_box_k3d(obb, color=0xff0000, opacity=0.3):
    c = obb['center']
    l = obb['lengths']
    d1, d2, d3 = obb['dir1']*l[0]/2, obb['dir2']*l[1]/2, obb['dir3']*l[2]/2

    # 8 corners
    corners = np.array([
        c - d1 - d2 - d3,
        c - d1 + d2 - d3,
        c + d1 - d2 - d3,
        c + d1 + d2 - d3,
        c - d1 - d2 + d3,
        c - d1 + d2 + d3,
        c + d1 - d2 + d3,
        c + d1 + d2 + d3,
    ], dtype=np.float32)

    # 12 triangles (2 per face)
    indices = np.array([
        0,1,2, 0,2,3,
        4,5,6, 4,6,7,
        0,1,5, 0,5,4,
        2,3,7, 2,7,6,
        1,2,6, 1,6,5,
        0,3,7, 0,7,4
    ], dtype=np.uint32)

    return k3d.mesh(corners, indices, color=color, opacity=opacity)

# ----------------------
# Example usage
# ----------------------
partnet_id = 1095  # obtained from syms/1.mat
obb_file = f'./dropbox/Chair/obbs/{partnet_id}.obb'

obbs = read_obb_file(obb_file)
plot = k3d.plot(grid_visible=True)

colors = [0xff0000, 0x00ff00, 0x0000ff, 0xffff00, 0xff00ff, 0x00ffff]

for i, obb in enumerate(obbs):
    plot += make_box_k3d(obb, color=colors[i % len(colors)])

plot.display()

ValueError: could not convert string to float: 'N'